# Laborator 8 – Arbori de Decizie pe setul de date Wine

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Ex 1 – Incarcam setul de date
wine = load_wine(as_frame=True)
print("Primele 5 randuri:")
print(wine.frame.head())

In [ ]:
# Ex 2 – Caracteristici disponibile
print("Caracteristici:")
for f in wine.feature_names:
    print(f"  {f}")
print(f"\nClase: {wine.target_names.tolist()}")

In [ ]:
# Ex 3a – Arbore de decizie cu 2 caracteristici (alcohol, flavanoids), max_depth=2
X2 = wine.frame[['alcohol','flavanoids']].values
y  = wine.target.values
X_tr, X_te, y_tr, y_te = train_test_split(X2, y, test_size=0.2, random_state=42)

tree2 = DecisionTreeClassifier(max_depth=2, random_state=42)
tree2.fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(10, 5))
plot_tree(tree2, feature_names=['alcohol','flavanoids'],
          class_names=wine.target_names, filled=True, ax=ax)
plt.title('Arbore de decizie (max_depth=2, alcohol + flavanoids)')
plt.tight_layout()
plt.savefig('lab8_tree2.png', dpi=80)
plt.show()
print(f"Acuratete (max_depth=2): {accuracy_score(y_te, tree2.predict(X_te)):.4f}")

In [ ]:
# Ex 3b – Interpretare primele noduri
print("Interpretare arbore:")
print("  Nodul radacina: prima conditie de split (ex: flavanoids <= prag)")
print("  In functie de aceasta conditie, datele sunt separate in doua subseturi.")
print("  Frunzele prezic clasa majoritara din acel subset.")
print(f"  Prag radacina pe flavanoids: {tree2.tree_.threshold[0]:.4f}")

In [ ]:
# Ex 4 – Arbore complet (max_depth=None) pe 2 caracteristici
tree_full = DecisionTreeClassifier(max_depth=None, random_state=42)
tree_full.fit(X_tr, y_tr)
acc_full = accuracy_score(y_te, tree_full.predict(X_te))
print(f"Acuratete arbore complet (2 caracteristici): {acc_full:.4f}")

In [ ]:
# Ex 5 – Arbore pe toate 13 caracteristici + importanta
X_all = wine.data.values
X_tr_all, X_te_all, y_tr_all, y_te_all = train_test_split(X_all, y, test_size=0.2, random_state=42)

tree_all = DecisionTreeClassifier(max_depth=None, random_state=42)
tree_all.fit(X_tr_all, y_tr_all)
print(f"Acuratete (13 caracteristici): {accuracy_score(y_te_all, tree_all.predict(X_te_all)):.4f}")
print()
print("Importanta caracteristicilor:")
importances = pd.Series(tree_all.feature_importances_, index=wine.feature_names)
importances = importances.sort_values(ascending=False)
for feat, imp in importances.items():
    print(f"  {feat:30s}: {imp:.4f}")

In [ ]:
# Vizualizare importanta caracteristici
fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Importanta caracteristicilor – Wine Dataset')
ax.set_ylabel('Importanta')
ax.set_xlabel('Caracteristica')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('lab8_importances.png', dpi=80)
plt.show()

In [ ]:
# Raport clasificare
y_pred_all = tree_all.predict(X_te_all)
print("Raport clasificare:")
print(classification_report(y_te_all, y_pred_all, target_names=wine.target_names))

In [ ]:
# Bonus – Calcul manual Gini impurity pentru nodul radacina (subset de 6 exemple)
subset_X = X_all[:6, [0, 6]]   # alcohol, flavanoids
subset_y = y[:6]
classes, counts = np.unique(subset_y, return_counts=True)
n_total = len(subset_y)
gini = 1 - sum((c/n_total)**2 for c in counts)
print("=== Bonus: Gini Impurity manual ===")
print(f"Subset y: {subset_y}")
print(f"Clase: {classes}, Contor: {counts}")
print(f"Gini impurity nod radacina: {gini:.4f}")
print()
# Propunem un split pe alcohol > 13.0
split_val = 13.0
mask_left  = subset_X[:, 0] <= split_val
mask_right = subset_X[:, 0] >  split_val
def gini_node(labels):
    if len(labels) == 0: return 0
    _, cnts = np.unique(labels, return_counts=True)
    n = len(labels)
    return 1 - sum((c/n)**2 for c in cnts)
g_left  = gini_node(subset_y[mask_left])
g_right = gini_node(subset_y[mask_right])
w_left  = mask_left.sum()  / n_total
w_right = mask_right.sum() / n_total
gini_split = w_left * g_left + w_right * g_right
print(f"Split alcohol > {split_val}:")
print(f"  Stanga ({mask_left.sum()} ex): Gini={g_left:.4f}")
print(f"  Dreapta ({mask_right.sum()} ex): Gini={g_right:.4f}")
print(f"  Gini ponderat dupa split: {gini_split:.4f}")
print(f"  Information Gain: {gini - gini_split:.4f}")
print("  -> Split {'bun' if gini > gini_split else 'slab'} (IG > 0 inseamna split util)")
